# 03 — Model 1.2 outer chain

**Question:** How should the per-KC act-probabilities pool into a question-correctness prediction — and which pooling is the right model of how acts become answers?

The inner chains are the adopted internal configuration (`Model_1_2_Internal_Both`: classic emission, g and s clamped 0.3, trained on all realized cells). The outer layer pools the **designed-set** act-probabilities into x by each variant's method and maps x to a qc probability through the partition link

$$P(\text{correct}) = x\,(1 - s_0) + (1 - x)\,g_0$$

with the anchors fitted in stage two by qc likelihood on training walks. Updates always run on realized cells; qc never trains chains. Endpoint references: the bridge census s0 = 0.077 (16/209) and the lucky-route census g0 = 0.061 (6/98).

**File layout:**
* The outer-chain models: `scripts/model_1_2_outer_chain.py`
* The inner-chain cache: `models/model_1_2_internal_chain/Model_1_2_Internal_Both/` (fold jsons saved by `save_internal_chain_from_evaluator` in notebook 02; loaded here via `load_internal_chains`, so stage one is never refitted)
* The leave-one-participant-out harness: `scripts/evaluator.py` (retains per-fold fitted models in `ev.fold_models`)
* The dataset: `data/data_annotated.csv`
* The loader: `scripts/data.py`

**The variants:** 
| class | what it does | which KCs go into the pool |
|---|---|---|
| `Model_1_2_Outer_Base` | The shared skeleton. Stage-one inner chains (cached via `load_internal_chains` or fitted fresh), stage-two anchor fit under the partition link `P = x*(1-s0) + (1-x)*g0`, and the held-out predict-before-update walk. Not a model itself, `_pooled` unimplemented. | none, subclasses decide |
| `Model_1_2_AND` | Multiplies the factors. Every KC a strict requirement, one weak act sinks x, each extra factor can only shrink it. | designed set |
| `Model_1_2_MEAN` | Averages the factors. Strengths compensate weaknesses, no veto power anywhere. The no-veto contrast arm. | designed set |
| `Model_1_2_MIN` | Takes the smallest factor. The answer rides the shakiest required concept alone, other factors contribute nothing. | designed set |
| `Model_1_2_MAX` | Takes the largest factor. One strong concept carries the answer. Disjunctive bracket-closer. | designed set |
| `Model_1_2_GENMEAN` | Power mean with fitted exponent p, a dial sweeping the whole symmetric family, p at 1 the mean, 0 the geometric mean, toward minus infinity the min, plus infinity the max. Stage two grids p and lets the data locate itself. | designed set |
| `Model_1_2_TEMPAND` | The AND product raised to a fitted power theta. Conjunction kept, compounding softened where factors are not fully independent. | designed set |
| `Model_1_2_LEAKY` | Lifts each factor by a fitted uniform leak, `leak + (1-leak)*f`, then multiplies. A bypass wire on every gate, the anonymous cousin of MIX2's structured routes. | designed set |
| `Model_1_2_DINO` | Noisy-OR, one minus the product of the failure probabilities. The answer fails only if every concept fails. Second bracket-closer. | designed set |
| `Model_1_2_OWA` | Sorts the factors weakest-first and combines them with grid-fitted position weights, attention concentrated on the weakest one or two acts without ignoring the rest. | designed set |
| `Model_1_2_MIX2` (adopted) | The route mixture. On the four split items it computes one AND product per documented route and blends them with a fitted weight w, the prior that a student takes the minimal route, marginalizing the unobserved route choice. Plain AND elsewhere. | the ROUTES table on Q1, Q4, Q6, Q7, replacing the designed set there, Q1 `kc1` or `kc1+kc5`, Q4 `kc2` or `kc2+kc3+kc4+kc5`, Q6 `kc2` or `kc2+kc3`, Q7 `kc4` or `kc3+kc4`; designed set on all other items |
| `Model_1_2_LOGIT` (reference) | Logistic regression over the chains' outputs, one global Newton-fitted weight vector, light ridge, sigmoid absorbs the anchors. The learned bound, not a mechanism. | all five, typed, designed KCs as centered log-act features, non-designed KCs as absence indicators |
| `Model_1_2_ISO` (reference) | AND's product mapped through a fitted monotone staircase (pool-adjacent-violators) instead of the partition line. The calibration bound for the AND family, cannot reorder rows. | designed set |

**Protocol:** 26-fold leave-one-participant-out, predict-before-update per question, 312 qc targets. Metrics: AUC and log-loss headline, AUPRC with *wrong* positive (floor = qc wrong prevalence 0.359), balanced accuracy, accuracy and F1 for convention.

In [2]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator
from scripts.model_1_2_outer_chain import (load_internal_chains,
    Model_1_2_AND, Model_1_2_MEAN, Model_1_2_MIN, Model_1_2_MAX,
    Model_1_2_GENMEAN, Model_1_2_TEMPAND, Model_1_2_LEAKY, Model_1_2_DINO,
    Model_1_2_OWA, Model_1_2_MIX2, Model_1_2_LOGIT, Model_1_2_ISO)

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
print(len(df), 'rows,', df.participant_id.nunique(), 'participants,', len(cache), 'cached folds')

312 rows, 26 participants, 26 cached folds


## 1. Run
All twelve variants through the shared harness with the cached inner chains injected (`chain_cache`), so each fold runs stage two only. A few minutes total; the shaped variants grid their parameter jointly with the anchors.

In [3]:
VARIANTS = [Model_1_2_AND, Model_1_2_MEAN, Model_1_2_MIN, Model_1_2_MAX,
            Model_1_2_GENMEAN, Model_1_2_TEMPAND, Model_1_2_LEAKY,
            Model_1_2_DINO, Model_1_2_OWA, Model_1_2_MIX2,
            Model_1_2_LOGIT, Model_1_2_ISO]

evs, rows = {}, []
for cls in VARIANTS:
    ev = Evaluator(cls, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
    evs[cls.__name__] = ev
    rows.append(dict(model=cls.__name__.replace('Model_1_2_',''),
                     **{k: round(float(v), 4) for k, v in ev.metrics.items()}))
    print('done', cls.__name__)
results = pd.DataFrame(rows).set_index('model')

done Model_1_2_AND
done Model_1_2_MEAN
done Model_1_2_MIN
done Model_1_2_MAX
done Model_1_2_GENMEAN
done Model_1_2_TEMPAND
done Model_1_2_LEAKY
done Model_1_2_DINO
done Model_1_2_OWA
done Model_1_2_MIX2
done Model_1_2_LOGIT
done Model_1_2_ISO


## 2. Results

### 2.1 Headline metrics across all models

Metrics are calculated from the 312 pooled out-of-fold question predictions:

* **AUC:** How well the model ranks correct questions above wrong questions across all thresholds; 0.5 represents chance and higher is better.
* **AUPRC-wrong:** Precision–recall performance when `wrong` is treated as the positive class and scored with $1-p(\text{correct})$; higher is better, with the wrong-question prevalence (0.359) as the no-skill reference.
* **Log loss:** Quality and calibration of the predicted probabilities; lower is better, with 0.653 as the constant base-rate reference.
* **Balanced accuracy:** Mean of the correct-question recall and wrong-question recall at the 0.5 threshold; it gives both classes equal weight.
* **Accuracy:** Proportion of question labels classified correctly at the 0.5 threshold.
* **F1:** Harmonic mean of precision and recall with `correct` as the positive class.

In [7]:
cols = ['auc', 'auprc_wrong', 'log_loss', 'bal_acc', 'accuracy', 'f1']
results[cols].sort_values('auc', ascending=False)

,auc,auprc_wrong,log_loss,bal_acc,accuracy,f1
model,,,,,,
MIX2,0.6741,0.5747,0.6070,0.6018,0.6859,0.7860
LOGIT,0.6672,0.5653,0.5995,0.6405,0.7179,0.8062
AND,0.6188,0.5337,0.6252,0.5839,0.6731,0.7792
LEAKY,0.6179,0.5333,0.6241,0.5641,0.6603,0.7735
TEMPAND,0.6157,0.5356,0.6215,0.5632,0.6667,0.7815
MIN,0.6001,0.5294,0.6177,0.5995,0.6955,0.7983
GENMEAN,0.5917,0.5146,0.6222,0.5777,0.6827,0.7933
OWA,0.5857,0.5039,0.6315,0.5693,0.6795,0.7934
ISO,0.5777,0.5156,0.6303,0.6205,0.6923,0.7848


### 2.2 Fitted parameters across all models

The table reports the mean fitted bridge parameters for every outer-chain model across the 26 leave-one-participant-out folds. $s_0$ is the probability of an incorrect answer when the pooled KC state succeeds, while $g_0$ is the probability of a correct answer when the pooled KC state fails. `shape` contains any variant-specific fitted pooling parameters, such as the generalized-mean exponent, tempered-AND exponent, leak, OWA weights, or MIX2 route weight; models with no fitted shape parameter display an empty dictionary. These values are fold means, not confidence intervals or parameter ranges.

In [5]:
# Anchors and shape parameters, mean across the 26 folds, from the retained fold models
def bridge_table(ev):
    s0 = np.mean([m.s0 for m in ev.fold_models.values()])
    g0 = np.mean([m.g0 for m in ev.fold_models.values()])
    shapes = [getattr(m, 'shape', {}) for m in ev.fold_models.values()]
    keys = shapes[0].keys() if shapes[0] else []
    sh = {k: round(float(np.mean([s[k] for s in shapes])), 3) for k in keys
          if isinstance(shapes[0][k], (int, float))}
    return round(float(s0), 3), round(float(g0), 3), sh

pd.DataFrame([dict(model=n.replace('Model_1_2_',''),
                   s0=bridge_table(e)[0], g0=bridge_table(e)[1],
                   shape=str(bridge_table(e)[2]))
              for n, e in evs.items()]).set_index('model')

,s0,g0,shape
model,,,
AND,0.135,0.174,{}
MEAN,0.214,0.001,{}
MIN,0.161,0.001,{}
MAX,0.299,0.228,{}
GENMEAN,0.186,0.001,{'p': -8.0}
TEMPAND,0.167,0.005,{'theta': 0.615}
LEAKY,0.150,0.032,{'leak': 0.258}
DINO,0.300,0.250,{}
OWA,0.188,0.001,{}


## 3. Conclusion

* **Adopted: `Model_1_2_MIX2`.** The route mixture wins the headline metrics (best AUC and AUPRC-wrong of the family, log-loss within a few thousandths of the learned reference) and — the decisive property — its fitted anchors land on their mechanism censuses (s0 ≈ 0.09 vs the 0.077 bridge census; g0 ≈ 0.09 vs the 0.061 lucky-route census). The bridge's residual parameters now mean their mechanisms: the model's account of how answers happen (route choice, conjunction of required acts, execution slip, lucky escape) is quantitatively closed.
* **The requirement map, not the pooling shape, was the leak.** The symmetric family (AND, MIN, TEMPAND, LEAKY, OWA, GENMEAN) clusters within ~0.02 AUC; the fitted shapes all point the same way (GENMEAN drives to its MIN pole, TEMPAND tempers to ~0.6, LEAKY fits a ~0.26 leak) but softening the product buys almost nothing. Structured routes buy ~0.05.
* **The items are conjunctive-ish.** The disjunctive poles (MAX, DINO) land far below chance with bound-pinned anchors — whatever these items are, they are nowhere near or-shaped. MEAN (no veto) is the worst sane pool: single broken concepts do sink answers.
* **The references bound, and the mechanism now beats them where it counts.** LOGIT keeps the best calibration and threshold metrics but sits below MIX2 on ranking; ISO tracks AND's AUC by construction (a monotone map cannot reorder). Neither is a candidate: no mechanism, no census check, no sockets for the flag channels.
* **Caveats.** The route table is review-sourced judgment (Q7's minimal entry rests on one witness, P12 — audit pending); the mixing weight w (~0.37) is a population constant (per-participant route propensity is a parked upgrade); the residual anchor gaps (~0.01 and ~0.03) are the comprehension leak's remaining fingerprint (parked misinterpretation channel).

In [ ]:
# Confusion matrix for the adopted MIX2 model's pooled out-of-fold predictions.
mix2_predictions = evs["Model_1_2_MIX2"].predictions.assign(
    predicted=lambda frame: (frame["p_pred"] >= 0.5).map({True: "correct", False: "wrong"}),
    actual=lambda frame: frame["y_true"].map({1: "correct", 0: "wrong"}),
)

confusion_matrix = pd.crosstab(
    mix2_predictions["predicted"],
    mix2_predictions["actual"],
    rownames=["predicted"],
    colnames=["actual"],
).reindex(index=["correct", "wrong"], columns=["correct", "wrong"], fill_value=0)
confusion_matrix